In [10]:
#Só é preciso executar ESTA CÉLULA... As restantes são apenas para visualizar os resultados

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, lower, lit, when, avg, coalesce, concat
from delta import configure_spark_with_delta_pip

# ===================================================
# 1. SETUP
# ===================================================
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"

builder = SparkSession.builder \
    .appName("Gold_Fact_Classificacao_Filmes") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()

# ===================================================
# 2. CARREGAR FONTES (Dimensão e Silvers)
# ===================================================
print("🔄 A carregar dados...")

# --- ALTERAÇÃO AQUI ---
# 1. Carregamos a Dimensão COMPLETA para ter o Título Original guardado
df_dim_base = spark.read.format("delta").load(f"{warehouse_location}/gold.db/dim_filme")

# 2. Criamos a versão de "Matching" (só IDs e Títulos normalizados) para os Joins funcionarem
df_dim_filme = df_dim_base.select("id_filme", trim(lower(col("title"))).alias("match_title"))
# ----------------------

# As Fontes Silver (Ratings)
df_amazon = spark.read.format("delta").load(f"{warehouse_location}/silver.db/titles") \
    .select(trim(lower(col("title"))).alias("t_ama"), col("release_year").alias("y_ama"), col("imdb_score"))
clas
df_netflix = spark.read.format("delta").load(f"{warehouse_location}/silver.db/netflix_titles") \
    .select(trim(lower(col("title"))).alias("t_net"), col("release_year").alias("y_net"), col("imdb_score"))

df_rotten = spark.read.format("delta").load(f"{warehouse_location}/silver.db/rotten_tomatoes") \
    .select(trim(lower(col("movie_title"))).alias("t_rot"), col("release_year").alias("y_rot"), col("tomatometer_rating"), col("audience_rating"))

# ===================================================
# 3. PREPARAR CADA FONTE (Normalização)
# ===================================================
print("🛠️ A normalizar escalas (0-100)...")

# Amazon
fact_amazon = df_amazon.join(df_dim_filme, df_amazon.t_ama == df_dim_filme.match_title) \
    .select(
        col("id_filme"),
        col("y_ama").alias("ano"),
        (col("imdb_score") * 10).alias("score_amazon")
    ).filter(col("score_amazon").isNotNull())

# Netflix
fact_netflix = df_netflix.join(df_dim_filme, df_netflix.t_net == df_dim_filme.match_title) \
    .select(
        col("id_filme"),
        col("y_net").alias("ano"),
        (col("imdb_score") * 10).alias("score_netflix")
    ).filter(col("score_netflix").isNotNull())

# Rotten
fact_rotten = df_rotten.join(df_dim_filme, df_rotten.t_rot == df_dim_filme.match_title) \
    .select(
        col("id_filme"),
        col("y_rot").alias("ano"),
        col("tomatometer_rating").alias("score_rotten_critics"),
        col("audience_rating").alias("score_rotten_audience")
    )

# ===================================================
# 4. UNIFICAR TUDO NA FACT TABLE (FULL JOIN)
# ===================================================
print("🔗 A consolidar a Tabela de Factos...")

# Join 1: Amazon + Netflix
df_join_1 = fact_amazon.join(fact_netflix, ["id_filme"], "full_outer") \
    .select(
        col("id_filme"),
        coalesce(fact_amazon.ano, fact_netflix.ano).alias("ano_temp"),
        col("score_amazon"),
        col("score_netflix")
    )

# Join 2: + Rotten Tomatoes
df_final = df_join_1.join(fact_rotten, ["id_filme"], "full_outer") \
    .select(
        col("id_filme"),
        # Chave Calendário
        when(
            coalesce(df_join_1.ano_temp, fact_rotten.ano).isNotNull(),
            lit("C")
        ).alias("prefix"),
        coalesce(df_join_1.ano_temp, fact_rotten.ano).alias("ano_final"),
        
        # Métricas
        col("score_amazon"),
        col("score_netflix"),
        col("score_rotten_critics"),
        col("score_rotten_audience")
    )

# Formatar id_calendario corretamente
df_final = df_final.withColumn("id_calendario", concat(col("prefix"), col("ano_final").cast("string"))) \
    .drop("prefix", "ano_final")

print("🏷️ A adicionar os nomes dos filmes...")
# Fazemos um Left Join com a tabela base que carregámos no início para ir buscar o 'title'
df_final_com_nome = df_final.join(df_dim_base.select("id_filme", "title"), "id_filme", "left")

# Reordenamos as colunas para o título aparecer logo a seguir ao ID
df_final_final = df_final_com_nome.select(
    "id_filme",
    "title",  # <--- O nome está aqui!
    "id_calendario",
    "score_amazon",
    "score_netflix",
    "score_rotten_critics",
    "score_rotten_audience"
)
# ----------------------------------------

# ===================================================
# 5. GRAVAR GOLD
# ===================================================
fact_path = f"{warehouse_location}/gold.db/fact_classificacao_filmes"
print(f"💾 A gravar em: {fact_path}")

df_final_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(fact_path)

spark.sql(f"CREATE DATABASE IF NOT EXISTS gold LOCATION '{warehouse_location}/gold.db'")
spark.sql(f"CREATE TABLE IF NOT EXISTS gold.fact_classificacao_filmes USING DELTA LOCATION '{fact_path}'")

print("🎉 SUCESSO! Tabela 'fact_classificacao_filmes' criada com NOMES.")
print("Esta tabela permite comparar: Amazon (IMDb) vs Netflix (IMDb) vs Rotten (Critics/Audience).")
df_final_final.orderBy("score_amazon", ascending=False).show(10, truncate=False)

🔄 A carregar dados...
🛠️ A normalizar escalas (0-100)...
🔗 A consolidar a Tabela de Factos...
🏷️ A adicionar os nomes dos filmes...
💾 A gravar em: hdfs://hdfs-nn:9000/warehouse/gold.db/fact_classificacao_filmes
🎉 SUCESSO! Tabela 'fact_classificacao_filmes' criada com NOMES.
Esta tabela permite comparar: Amazon (IMDb) vs Netflix (IMDb) vs Rotten (Critics/Audience).
+--------+-------------------------+-------------+------------+-------------+--------------------+---------------------+
|id_filme|title                    |id_calendario|score_amazon|score_netflix|score_rotten_critics|score_rotten_audience|
+--------+-------------------------+-------------+------------+-------------+--------------------+---------------------+
|F20383  |Pawankhind               |C2022        |99.0        |null         |null                |null                 |
|F33429  |Water Helps the Blood Run|C2019        |97.0        |null         |null                |null                 |
|F6551   |Couple of Mirrors 

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# ===================================================
# 1. SETUP
# ===================================================
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"
path_gold_fact = f"{warehouse_location}/gold.db/fact_classificacao_filmes"
path_gold_dim = f"{warehouse_location}/gold.db/dim_filme"

df_fact = spark.read.format("delta").load(path_gold_fact)
df_dim = spark.read.format("delta").load(path_gold_dim)

df_completo = df_fact.join(df_dim, "id_filme", "left") \
    .select("title", "score_amazon", "score_netflix", "score_rotten_critics", "score_rotten_audience")

# ===================================================
# 2. PROCURAR INTERSEÇÕES (As "Agulhas no Palheiro")
# ===================================================

print("\n=== 🔎 PROVA 1: Filmes na AMAZON que também têm ROTTEN TOMATOES ===")
# Filtra apenas onde ambos têm valor
df_completo.filter(
    col("score_amazon").isNotNull() & 
    col("score_rotten_critics").isNotNull()
).select("title", "score_amazon", "score_rotten_critics").show(10, truncate=False)

print("\n=== 🔎 PROVA 2: Filmes na NETFLIX que também têm ROTTEN TOMATOES ===")
df_completo.filter(
    col("score_netflix").isNotNull() & 
    col("score_rotten_critics").isNotNull()
).select("title", "score_netflix", "score_rotten_critics").show(10, truncate=False)


=== 🔎 PROVA 1: Filmes na AMAZON que também têm ROTTEN TOMATOES ===
+---------------------+------------+--------------------+
|title                |score_amazon|score_rotten_critics|
+---------------------+------------+--------------------+
|For Pete's Sake      |62.0        |57.0                |
|Forces of Nature     |54.0        |45.0                |
|Forever Mine         |52.0        |29.0                |
|Forget Me Not        |65.0        |67.0                |
|Forgiving Dr. Mengele|71.0        |90.0                |
|Forsaken             |63.0        |42.0                |
|Four                 |58.0        |67.0                |
|Four Lions           |72.0        |83.0                |
|Foxy Brown           |65.0        |60.0                |
|FrackNation          |60.0        |50.0                |
+---------------------+------------+--------------------+
only showing top 10 rows


=== 🔎 PROVA 2: Filmes na NETFLIX que também têm ROTTEN TOMATOES ===
+---------------------+--

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number, count
from pyspark.sql.window import Window

# ===================================================
# 1. SETUP
# ===================================================
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"
path_gold_fact = f"{warehouse_location}/gold.db/fact_classificacao_filmes"
path_gold_dim = f"{warehouse_location}/gold.db/dim_filme"

df_fact = spark.read.format("delta").load(path_gold_fact)
df_dim = spark.read.format("delta").load(path_gold_dim)

# Juntar Nomes
df_completo = df_fact.join(df_dim, "id_filme", "left") \
    .select(
        "title", 
        "score_amazon", 
        "score_netflix", 
        "score_rotten_critics", 
        "score_rotten_audience"
    )

# ===================================================
# 2. LIMPEZA DE DUPLICADOS (O "Home" Fix)
# ===================================================
# Definimos uma regra: Se houver nomes repetidos, ficamos com aquele que tem
# a maior nota no Rotten Tomatoes (assumindo ser o filme "famoso")
window_spec = Window.partitionBy("title").orderBy(col("score_rotten_critics").desc())

df_limpo = df_completo.withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") == 1) \
    .drop("rank")

# ===================================================
# 3. MOSTRAR O "SANTO GRAAL" LIMPO
# ===================================================
print("\n=== 🏆 Filmes confirmados nas 3 plataformas ===")
# Filtramos apenas os que têm notas nas 3 colunas
df_santo_graal = df_limpo.filter(
    col("score_amazon").isNotNull() & (col("score_amazon") > 0) &
    col("score_netflix").isNotNull() & (col("score_netflix") > 0) &
    col("score_rotten_critics").isNotNull()
)

df_santo_graal.select("title", "score_amazon", "score_netflix", "score_rotten_critics") \
    .orderBy(col("score_rotten_critics").desc()) \
    .show(20, truncate=False)


=== 🏆 Filmes confirmados nas 3 plataformas ===
+-----------------------------------------+------------+-------------+--------------------+
|title                                    |score_amazon|score_netflix|score_rotten_critics|
+-----------------------------------------+------------+-------------+--------------------+
|City of Joy                              |65.0        |75.0         |100.0               |
|Conspiracy                               |46.0        |62.0         |100.0               |
|Heidi                                    |74.0        |71.0         |100.0               |
|Kabhi Khushi Kabhie Gham                 |74.0        |74.0         |100.0               |
|Mike Birbiglia: My Girlfriend's Boyfriend|80.0        |80.0         |100.0               |
|Sahara                                   |67.0        |55.0         |100.0               |
|The Black Godfather                      |44.0        |74.0         |100.0               |
|The Hospital                   